# FAME-Energy — Notebook 05 v2.1
## DOC Calibration Production — paralelização segura por \(\theta\)

Sim: o bloco pesado deve ser executado em paralelo, mas **a paralelização correta é por \(\theta\)**.

Para cada valor fixo:

\[
\theta_j:
\quad
d_1\rightarrow d_2\rightarrow\cdots\rightarrow d_{183}.
\]

Os dias da mesma trajetória não podem ser paralelizados porque o estado operacional é propagado:

\[
s_{d+1}(\theta_j)
=
f\!\left(s_d(\theta_j),x_d^\star(\theta_j)\right).
\]

As nove trajetórias de \(\theta\), entretanto, são independentes.

Esta versão usa até **3 trajetórias simultâneas**, checkpoint diário e resume seguro.

O Test continua selado.

### Perfil de execução — Intel Core i7-12700K

Esta versão foi ajustada para 12 núcleos físicos / 20 processadores lógicos:

- até **6 trajetórias de \(\theta\)** simultâneas;
- cada trajetória permanece sequencial no tempo;
- cada worker abre seu próprio processo CBC;
- evita paralelização aninhada excessiva;
- checkpoints continuam sendo gravados diariamente;
- reinício do notebook retoma trajetórias incompletas.


In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, subprocess, shutil
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT=Path.cwd()
DOC_ROOT=ROOT/"data"/"fame_energy_doc_v21"
DOC_ROOT.mkdir(parents=True,exist_ok=True)

CORE_V13_DIR=ROOT/"data"/"fame_energy_core_v13"
DOC_V10_DIR=ROOT/"data"/"fame_energy_doc_v10"

CBC_EXE=Path(
    r"C:\Users\tiago\Dropbox\Artigos\FAME\FAME_IJDSA\Cbc-releases.2.10.13-windows-2025-msvs-v17-Release-x64\bin\cbc.exe"
)
if not CBC_EXE.is_file():
    raise FileNotFoundError(CBC_EXE)

MAX_PARALLEL_THETA=6
MAX_SOLVER_SECONDS=300
WORKER_WATCHDOG_SECONDS=6*3600

print("Logical CPU cores:",os.cpu_count())
print("Parallel theta trajectories:",MAX_PARALLEL_THETA)

In [ ]:
# Diagnóstico da estratégia multicore
LOGICAL_CPUS = os.cpu_count() or 1
RESERVED_LOGICAL_CPUS = max(2, LOGICAL_CPUS - 18)

if MAX_PARALLEL_THETA > len(THETA_GRID) if "THETA_GRID" in globals() else False:
    MAX_PARALLEL_THETA = len(THETA_GRID)

print("=" * 72)
print("FAME-Energy v2.1 — multicore execution profile")
print("=" * 72)
print("Logical CPUs detected :", LOGICAL_CPUS)
print("Theta workers          :", MAX_PARALLEL_THETA)
print("CBC processes max      :", MAX_PARALLEL_THETA)
print("Solver limit / solve   :", MAX_SOLVER_SECONDS, "s")
print("Worker watchdog        :", WORKER_WATCHDOG_SECONDS, "s")
print()
print("Estratégia:")
print("  paralelismo ENTRE trajetórias theta")
print("  sequência temporal DENTRO de cada theta")
print("  checkpoint diário + resume")
print("=" * 72)

## 1. Protocolo congelado e Test selado

In [ ]:
required=[
    DOC_V10_DIR/"temporal_protocol.csv",
    DOC_V10_DIR/"theta_grid_frozen.csv",
    DOC_V10_DIR/"protocol_freeze_manifest.csv",
    DOC_V10_DIR/"protocol.json",
]
missing=[str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("\n".join(missing))

protocol=pd.read_csv(DOC_V10_DIR/"temporal_protocol.csv")
theta_df=pd.read_csv(DOC_V10_DIR/"theta_grid_frozen.csv")
freeze=pd.read_csv(DOC_V10_DIR/"protocol_freeze_manifest.csv")

THETA_GRID=np.array(theta_df["theta"],dtype=float)

cal_row=protocol.loc[protocol["block"]=="calibration"].iloc[0]
test_row=protocol.loc[protocol["block"]=="test"].iloc[0]

CAL_START=pd.Timestamp(cal_row["start"])
CAL_END=pd.Timestamp(cal_row["end"])
TEST_START=pd.Timestamp(test_row["start"])
TEST_END=pd.Timestamp(test_row["end"])

protocol_sha=hashlib.sha256(
    (DOC_V10_DIR/"protocol.json").read_bytes()
).hexdigest()

if protocol_sha != str(freeze.loc[0,"protocol_sha256"]):
    raise RuntimeError("Protocol hash mismatch.")

if not bool(freeze.loc[0,"test_is_sealed"]):
    raise RuntimeError("Test is not sealed.")

print("Protocol SHA:",protocol_sha)
print("Calibration:",CAL_START.date(),"->",CAL_END.date())
print("Test remains sealed:",TEST_START.date(),"->",TEST_END.date())
print("Theta grid:",THETA_GRID.tolist())

## 2. Estado comum congelado da v1.3

In [ ]:
COMMON_U_FILE=CORE_V13_DIR/"common_initial_commitment.npy"
COMMON_P_FILE=CORE_V13_DIR/"common_initial_dispatch.npy"
SANITY_FILE=CORE_V13_DIR/"warmstart_core_sanity_checks.csv"

for p in [COMMON_U_FILE,COMMON_P_FILE,SANITY_FILE]:
    if not p.exists():
        raise FileNotFoundError(p)

sanity=pd.read_csv(SANITY_FILE)
bool_cols=sanity.select_dtypes(include=["bool"]).columns
if len(bool_cols) and not sanity[bool_cols].all().all():
    raise RuntimeError("v1.3 sanity checks failed.")

print("v1.3 common units:",int(np.load(COMMON_U_FILE).sum()))
print("v1.3 common dispatch:",float(np.load(COMMON_P_FILE).sum()))

## 3. Worker autônomo com checkpoint/resume

In [ ]:
WORKER_SCRIPT=DOC_ROOT/"fame_energy_calibration_worker.py"
worker_code='\nimport argparse, json, time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nfrom pyomo.environ import (\n    ConcreteModel, RangeSet, Var, Binary, NonNegativeReals,\n    Constraint, Objective, minimize, SolverFactory, value\n)\n\nparser=argparse.ArgumentParser()\nparser.add_argument("--root",required=True)\nparser.add_argument("--outdir",required=True)\nparser.add_argument("--theta",type=float,required=True)\nparser.add_argument("--start",required=True)\nparser.add_argument("--end",required=True)\nparser.add_argument("--initial-u",required=True)\nparser.add_argument("--initial-p",required=True)\nparser.add_argument("--cbc",required=True)\nparser.add_argument("--solver-seconds",type=int,default=300)\nparser.add_argument("--watchdog-seconds",type=int,default=21600)\nargs=parser.parse_args()\n\nROOT=Path(args.root)\nOUT=Path(args.outdir)\nOUT.mkdir(parents=True,exist_ok=True)\nCBC_EXE=Path(args.cbc)\nTHETA=float(args.theta)\n\ncheckpoint_file=OUT/"checkpoint.csv"\nstate_u_file=OUT/"state_u.npy"\nstate_p_file=OUT/"state_p.npy"\nstatus_file=OUT/"status.json"\n\ngen_candidates=list((ROOT/"external").rglob("SourceData/gen.csv"))\nif not gen_candidates:\n    gen_candidates=list(ROOT.rglob("SourceData/gen.csv"))\nif not gen_candidates:\n    raise FileNotFoundError("SourceData/gen.csv")\n\nGEN_FILE=sorted(gen_candidates,key=lambda p:len(str(p)))[0]\nSOURCE_DATA=GEN_FILE.parent\nRTS_BASE=SOURCE_DATA.parent\n\nDA_FILE=list(RTS_BASE.rglob("Load/DAY_AHEAD_regional_Load.csv"))[0]\nRT_FILE=list(RTS_BASE.rglob("Load/REAL_TIME_regional_Load.csv"))[0]\n\ngen=pd.read_csv(GEN_FILE)\n\ndef find_optional_col(df,patterns):\n    cols=list(df.columns)\n    low=[str(c).lower() for c in cols]\n    hits=[]\n    for j,name in enumerate(low):\n        if all(p.lower() in name for p in patterns):\n            hits.append(cols[j])\n    return hits[0] if len(hits)==1 else None\n\nRAMP_UP_COL=(find_optional_col(gen,["ramp","up"]) or find_optional_col(gen,["ramp rate"]))\nRAMP_DOWN_COL=find_optional_col(gen,["ramp","down"])\nMIN_UP_COL=find_optional_col(gen,["min","up"])\nMIN_DOWN_COL=find_optional_col(gen,["min","down"])\n\nthermal_mask=~gen["Fuel"].astype(str).isin(["Wind","Solar","Storage","Hydro","CSP"])\nthermal=gen.loc[thermal_mask].copy()\nthermal=thermal[pd.to_numeric(thermal["PMax MW"],errors="coerce")>0].reset_index(drop=True)\n\ndef rts_cost_points(row):\n    x=np.array([\n        float(row["Output_pct_0"]),\n        float(row["Output_pct_1"]),\n        float(row["Output_pct_2"]),\n        float(row["Output_pct_3"]),\n    ])\n    hr0=float(row["HR_avg_0"])\n    if np.isnan(hr0):\n        hr0=0.0\n    if 0<hr0<=3412:\n        hr0=3412.0\n    inc=np.array([\n        float(row["HR_incr_1"]),\n        float(row["HR_incr_2"]),\n        float(row["HR_incr_3"]),\n    ])\n    y=np.zeros(4)\n    y[0]=hr0*x[0]\n    y[1]=y[0]+(x[1]-x[0])*inc[0]\n    y[2]=y[1]+(x[2]-x[1])*inc[1]\n    y[3]=y[2]+(x[3]-x[2])*inc[2]\n    pmax=float(row["PMax MW"])\n    fuel=float(row["Fuel Price $/MMBTU"])\n    return x*pmax,y*pmax*fuel/1000.0\n\ncurve_rows=[]\nfor i,row in thermal.iterrows():\n    mw,cost=rts_cost_points(row)\n    sh=pd.to_numeric(pd.Series([row["Start Heat Cold MBTU"]]),errors="coerce").iloc[0]\n    nf=pd.to_numeric(pd.Series([row["Non Fuel Start Cost $"]]),errors="coerce").iloc[0]\n    fuel=float(row["Fuel Price $/MMBTU"])\n    startup=(0 if pd.isna(sh) else sh*fuel)+(0 if pd.isna(nf) else nf)\n    for k in range(4):\n        curve_rows.append({\n            "g":i,"segment_point":k,"mw":mw[k],"cost":cost[k],\n            "startup_cost":startup\n        })\ncurves=pd.DataFrame(curve_rows)\n\nsegments=[]\nfor gidx,grp in curves.groupby("g"):\n    grp=grp.sort_values("segment_point")\n    for k in range(3):\n        p0=grp.iloc[k]["mw"]; p1=grp.iloc[k+1]["mw"]\n        c0=grp.iloc[k]["cost"]; c1=grp.iloc[k+1]["cost"]\n        width=max(0.0,p1-p0)\n        slope=(c1-c0)/width if width>1e-9 else 0.0\n        segments.append({\n            "g":gidx,"segment":k,"width_mw":width,\n            "marginal_cost":slope\n        })\nsegments=pd.DataFrame(segments)\n\nda=pd.read_csv(DA_FILE)\nrt=pd.read_csv(RT_FILE)\nids=["Year","Month","Day","Period"]\nareas=[c for c in da.columns if c not in ids]\nda["forecast_load"]=da[areas].sum(axis=1)\n\nrt_areas=[c for c in rt.columns if c not in ids]\nrt["actual_5m"]=rt[rt_areas].sum(axis=1)\nrt["hour"]=((rt["Period"]-1)//12)+1\nactual_h=(\n    rt.groupby(["Year","Month","Day","hour"],as_index=False)["actual_5m"].mean()\n      .rename(columns={"hour":"Period","actual_5m":"actual_load"})\n)\nload=da[ids+["forecast_load"]].merge(actual_h,on=ids,how="inner")\nload["timestamp"]=pd.to_datetime(dict(\n    year=load.Year,month=load.Month,day=load.Day\n))+pd.to_timedelta(load.Period-1,unit="h")\nload["date"]=load["timestamp"].dt.normalize()\n\ndef get_day(date):\n    date=pd.Timestamp(date).normalize()\n    d=load[load["date"]==date].sort_values("Period").copy()\n    if len(d)!=24 or d["Period"].nunique()!=24:\n        raise RuntimeError(f"Invalid day {date.date()}: {len(d)} rows.")\n    return d\n\ndef fame_transform(x,theta):\n    return np.asarray(x,dtype=float)*np.exp(float(theta))\n\ndef solve_daily_uc(decision_load_24,initial_commitment,initial_dispatch):\n    G=len(thermal); T=24; K=3\n    base={}; startup={}; p0={}; pmax={}\n    widths={}; slopes={}; ramp_up={}; ramp_down={}; min_up={}; min_down={}\n\n    for g in range(G):\n        cg=curves[curves.g==g].sort_values("segment_point")\n        sg=segments[segments.g==g].sort_values("segment")\n        p0[g]=float(cg.iloc[0].mw)\n        pmax[g]=float(thermal.loc[g,"PMax MW"])\n        base[g]=float(cg.iloc[0].cost)\n        startup[g]=float(cg.iloc[0].startup_cost)\n        for k in range(K):\n            widths[g,k]=float(sg.iloc[k].width_mw)\n            slopes[g,k]=float(sg.iloc[k].marginal_cost)\n\n        if RAMP_UP_COL is not None:\n            v=pd.to_numeric(pd.Series([thermal.loc[g,RAMP_UP_COL]]),errors="coerce").iloc[0]\n            ramp_up[g]=None if pd.isna(v) or v<=0 else float(v)\n        if RAMP_DOWN_COL is not None:\n            v=pd.to_numeric(pd.Series([thermal.loc[g,RAMP_DOWN_COL]]),errors="coerce").iloc[0]\n            ramp_down[g]=None if pd.isna(v) or v<=0 else float(v)\n        if MIN_UP_COL is not None:\n            v=pd.to_numeric(pd.Series([thermal.loc[g,MIN_UP_COL]]),errors="coerce").iloc[0]\n            min_up[g]=0 if pd.isna(v) or v<0 else int(np.ceil(float(v)))\n        if MIN_DOWN_COL is not None:\n            v=pd.to_numeric(pd.Series([thermal.loc[g,MIN_DOWN_COL]]),errors="coerce").iloc[0]\n            min_down[g]=0 if pd.isna(v) or v<0 else int(np.ceil(float(v)))\n\n    m=ConcreteModel()\n    m.G=RangeSet(0,G-1); m.T=RangeSet(0,T-1); m.K=RangeSet(0,K-1)\n    m.u=Var(m.G,m.T,domain=Binary)\n    m.y=Var(m.G,m.T,domain=Binary)\n    m.z=Var(m.G,m.T,domain=Binary)\n    m.seg=Var(m.G,m.K,m.T,domain=NonNegativeReals)\n\n    def seg_cap(m,g,k,t):\n        return m.seg[g,k,t] <= widths[g,k]*m.u[g,t]\n    m.seg_cap=Constraint(m.G,m.K,m.T,rule=seg_cap)\n\n    def startup_rule(m,g,t):\n        prev=initial_commitment[g] if t==0 else m.u[g,t-1]\n        return m.y[g,t] >= m.u[g,t]-prev\n    m.startup=Constraint(m.G,m.T,rule=startup_rule)\n\n    def shutdown_rule(m,g,t):\n        prev=initial_commitment[g] if t==0 else m.u[g,t-1]\n        return m.z[g,t] >= prev-m.u[g,t]\n    m.shutdown=Constraint(m.G,m.T,rule=shutdown_rule)\n\n    def prod(m,g,t):\n        return p0[g]*m.u[g,t]+sum(m.seg[g,k,t] for k in m.K)\n\n    def balance(m,t):\n        return sum(prod(m,g,t) for g in m.G) >= float(decision_load_24[t])\n    m.balance=Constraint(m.T,rule=balance)\n\n    if RAMP_UP_COL is not None:\n        def ru_rule(m,g,t):\n            ru=ramp_up.get(g)\n            if ru is None:\n                return Constraint.Skip\n            prev=initial_dispatch[g] if t==0 else prod(m,g,t-1)\n            return prod(m,g,t)-prev <= ru+pmax[g]*m.y[g,t]\n        m.ru=Constraint(m.G,m.T,rule=ru_rule)\n\n    if RAMP_DOWN_COL is not None:\n        def rd_rule(m,g,t):\n            rd=ramp_down.get(g)\n            if rd is None:\n                return Constraint.Skip\n            prev=initial_dispatch[g] if t==0 else prod(m,g,t-1)\n            return prev-prod(m,g,t) <= rd+pmax[g]*m.z[g,t]\n        m.rd=Constraint(m.G,m.T,rule=rd_rule)\n\n    if MIN_UP_COL is not None and MIN_DOWN_COL is not None:\n        def mu_rule(m,g,t):\n            U=min_up.get(g,0)\n            if U<=1:\n                return Constraint.Skip\n            start=max(0,t-U+1)\n            return sum(m.y[g,j] for j in range(start,t+1)) <= m.u[g,t]\n        m.mu=Constraint(m.G,m.T,rule=mu_rule)\n\n        def md_rule(m,g,t):\n            D=min_down.get(g,0)\n            if D<=1:\n                return Constraint.Skip\n            start=max(0,t-D+1)\n            return sum(m.z[g,j] for j in range(start,t+1)) <= 1-m.u[g,t]\n        m.md=Constraint(m.G,m.T,rule=md_rule)\n\n    m.obj=Objective(\n        expr=sum(\n            startup[g]*m.y[g,t]+base[g]*m.u[g,t]\n            +sum(slopes[g,k]*m.seg[g,k,t] for k in m.K)\n            for g in m.G for t in m.T\n        ),\n        sense=minimize\n    )\n\n    solver=SolverFactory("cbc",executable=str(CBC_EXE))\n    solver.options["seconds"]=float(args.solver_seconds)\n\n    t0=time.time()\n    res=solver.solve(m)\n    elapsed=time.time()-t0\n    term=str(res.solver.termination_condition).lower()\n\n    if term!="optimal":\n        return {"ok":False,"termination":term,"solve_seconds":elapsed}\n\n    u=np.array([[round(value(m.u[g,t])) for t in m.T] for g in m.G],dtype=int)\n    dispatch=np.zeros((G,T))\n    for g in range(G):\n        for t in range(T):\n            dispatch[g,t]=p0[g]*u[g,t]+sum(value(m.seg[g,k,t]) for k in range(K))\n\n    return {\n        "ok":True,\n        "termination":term,\n        "solve_seconds":elapsed,\n        "u":u,\n        "dispatch":dispatch,\n        "planned_cost":float(value(m.obj)),\n        "final_u":u[:,-1].copy(),\n        "final_p":dispatch[:,-1].copy()\n    }\n\ndef realized_day_cost(commitment,actual_load_24,voll=10000.0):\n    G,T=commitment.shape\n    total_gen=0.0; total_shed=0.0\n    for t in range(T):\n        active=np.where(commitment[:,t]==1)[0]\n        demand=float(actual_load_24[t])\n        blocks=[]; base_generation=0.0; base_cost=0.0\n        for g in active:\n            cg=curves[curves.g==g].sort_values("segment_point")\n            sg=segments[segments.g==g].sort_values("segment")\n            base_generation+=float(cg.iloc[0].mw)\n            base_cost+=float(cg.iloc[0].cost)\n            for _,r in sg.iterrows():\n                blocks.append((float(r.marginal_cost),float(r.width_mw)))\n        remaining=max(0.0,demand-base_generation)\n        var_cost=0.0\n        for mc,width in sorted(blocks,key=lambda x:x[0]):\n            if remaining<=1e-9:\n                break\n            q=min(width,remaining)\n            var_cost+=mc*q\n            remaining-=q\n        shed=max(0.0,remaining)\n        total_gen+=base_cost+var_cost\n        total_shed+=shed\n    return {\n        "generation_cost":total_gen,\n        "load_shedding":total_shed,\n        "total_realized_cost":total_gen+voll*total_shed\n    }\n\nstart_date=pd.Timestamp(args.start)\nend_date=pd.Timestamp(args.end)\nall_dates=list(pd.date_range(start_date,end_date,freq="D"))\n\nif checkpoint_file.exists() and state_u_file.exists() and state_p_file.exists():\n    ck=pd.read_csv(checkpoint_file)\n    if len(ck):\n        last=pd.Timestamp(ck["date"].iloc[-1])\n        dates=[d for d in all_dates if d>last]\n        state_u=np.load(state_u_file)\n        state_p=np.load(state_p_file)\n        rows=ck.to_dict("records")\n    else:\n        dates=all_dates\n        state_u=np.load(args.initial_u)\n        state_p=np.load(args.initial_p)\n        rows=[]\nelse:\n    dates=all_dates\n    state_u=np.load(args.initial_u)\n    state_p=np.load(args.initial_p)\n    rows=[]\n\nworker_start=time.time()\n\nfor idx,date in enumerate(dates,1):\n    if time.time()-worker_start > args.watchdog_seconds:\n        status={\n            "theta":THETA,"status":"watchdog_timeout",\n            "last_completed_date":rows[-1]["date"] if rows else None\n        }\n        status_file.write_text(json.dumps(status,indent=2),encoding="utf-8")\n        raise RuntimeError("Worker watchdog exceeded.")\n\n    d=get_day(date)\n    decision=fame_transform(d["forecast_load"].values,THETA)\n\n    print(\n        f"theta={THETA:+.3f} | {date.date()} | {idx}/{len(dates)}",\n        flush=True\n    )\n\n    sol=solve_daily_uc(decision,state_u,state_p)\n\n    if not sol["ok"]:\n        status={\n            "theta":THETA,"status":"solver_failure",\n            "date":str(date.date()),\n            "termination":sol["termination"],\n            "solve_seconds":sol["solve_seconds"]\n        }\n        status_file.write_text(json.dumps(status,indent=2),encoding="utf-8")\n        raise RuntimeError(str(status))\n\n    real=realized_day_cost(sol["u"],d["actual_load"].values)\n\n    rows.append({\n        "theta":THETA,\n        "date":str(date.date()),\n        "planned_cost":sol["planned_cost"],\n        "generation_cost":real["generation_cost"],\n        "load_shedding":real["load_shedding"],\n        "total_realized_cost":real["total_realized_cost"],\n        "solve_seconds":sol["solve_seconds"],\n        "start_units":int(state_u.sum()),\n        "end_units":int(sol["final_u"].sum())\n    })\n\n    pd.DataFrame(rows).to_csv(checkpoint_file,index=False)\n    np.save(state_u_file,sol["final_u"])\n    np.save(state_p_file,sol["final_p"])\n\n    state_u=sol["final_u"].copy()\n    state_p=sol["final_p"].copy()\n\nstatus={\n    "theta":THETA,\n    "status":"completed",\n    "n_days":len(rows),\n    "last_completed_date":rows[-1]["date"] if rows else None,\n    "mean_realized_cost":float(pd.DataFrame(rows)["total_realized_cost"].mean())\n}\nstatus_file.write_text(json.dumps(status,indent=2),encoding="utf-8")\nprint(json.dumps(status),flush=True)\n'
WORKER_SCRIPT.write_text(worker_code,encoding="utf-8")
print("Worker written:",WORKER_SCRIPT)

## 4. Estado inicial comum da Calibration

Antes de paralelizar \(\theta\), propagamos uma única trajetória baseline \(\theta=0\) pelo Development,
de 02/01/2020 até 31/03/2020.

Essa etapa é sequencial e é feita apenas uma vez. Depois dela, todas as trajetórias de Calibration
começam exatamente do mesmo estado.

In [ ]:
CAL_STATE_DIR=DOC_ROOT/"calibration_initial_state"
CAL_STATE_DIR.mkdir(parents=True,exist_ok=True)

CAL_U=CAL_STATE_DIR/"calibration_initial_u.npy"
CAL_P=CAL_STATE_DIR/"calibration_initial_p.npy"

DEV_PROP_START=pd.Timestamp("2020-01-02")
DEV_PROP_END=CAL_START-pd.Timedelta(days=1)

if CAL_U.exists() and CAL_P.exists():
    print("Calibration initial state already exists; skipping propagation.")
else:
    dev_worker_dir=DOC_ROOT/"development_baseline_state"
    dev_worker_dir.mkdir(parents=True,exist_ok=True)

    cmd=[
        sys.executable,"-u",str(WORKER_SCRIPT),
        "--root",str(ROOT),
        "--outdir",str(dev_worker_dir),
        "--theta","0.0",
        "--start",str(DEV_PROP_START.date()),
        "--end",str(DEV_PROP_END.date()),
        "--initial-u",str(COMMON_U_FILE),
        "--initial-p",str(COMMON_P_FILE),
        "--cbc",str(CBC_EXE),
        "--solver-seconds",str(MAX_SOLVER_SECONDS),
        "--watchdog-seconds",str(WORKER_WATCHDOG_SECONDS),
    ]

    print("Propagating baseline through Development...")
    rc=subprocess.run(cmd).returncode
    if rc!=0:
        raise RuntimeError(
            "Development propagation failed. "
            "Inspect data/fame_energy_doc_v21/development_baseline_state/status.json"
        )

    shutil.copy2(dev_worker_dir/"state_u.npy",CAL_U)
    shutil.copy2(dev_worker_dir/"state_p.npy",CAL_P)

print("Calibration initial units:",int(np.load(CAL_U).sum()))
print("Calibration initial dispatch:",float(np.load(CAL_P).sum()))

### Por que `ThreadPoolExecutor` continua correto aqui?

O executor **não resolve os MILPs em threads Python**. Cada tarefa chama
`subprocess.run(...)`, criando um processo Python worker independente e,
dentro dele, um processo `cbc.exe`.

Portanto, o trabalho computacional pesado ocorre fora do GIL do Python.
Com `MAX_PARALLEL_THETA = 6`, podem existir até seis processos CBC
independentes usando diferentes núcleos do processador.

Não paralelizamos os dias de uma mesma trajetória, pois o estado final
do dia \(d\) é condição inicial do dia \(d+1\).

## 5. Calibration paralela por \(\theta\)

Cada worker percorre 183 dias sequencialmente. Até seis workers são executados simultaneamente.

Se uma execução for interrompida, rode novamente esta célula/notebook: o worker continua do último
checkpoint, sem repetir os dias concluídos.

In [ ]:
CAL_ROOT=DOC_ROOT/"calibration_workers"
CAL_ROOT.mkdir(parents=True,exist_ok=True)

def theta_label(theta):
    return f"theta_{theta:+.3f}".replace("+","p").replace("-","m")

def launch_theta(theta):
    outdir=CAL_ROOT/theta_label(theta)
    outdir.mkdir(parents=True,exist_ok=True)

    cmd=[
        sys.executable,"-u",str(WORKER_SCRIPT),
        "--root",str(ROOT),
        "--outdir",str(outdir),
        "--theta",str(float(theta)),
        "--start",str(CAL_START.date()),
        "--end",str(CAL_END.date()),
        "--initial-u",str(CAL_U),
        "--initial-p",str(CAL_P),
        "--cbc",str(CBC_EXE),
        "--solver-seconds",str(MAX_SOLVER_SECONDS),
        "--watchdog-seconds",str(WORKER_WATCHDOG_SECONDS),
    ]

    with open(outdir/"stdout.txt","a",encoding="utf-8") as fout, \
         open(outdir/"stderr.txt","a",encoding="utf-8") as ferr:
        proc=subprocess.run(cmd,stdout=fout,stderr=ferr)

    return {
        "theta":float(theta),
        "returncode":proc.returncode,
        "outdir":str(outdir)
    }

worker_results=[]

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_THETA) as executor:
    futures={
        executor.submit(launch_theta,theta):theta
        for theta in THETA_GRID
    }

    for future in as_completed(futures):
        theta=futures[future]
        try:
            result=future.result()
        except Exception as exc:
            result={
                "theta":float(theta),
                "returncode":None,
                "error":repr(exc)
            }

        worker_results.append(result)
        print("FINISHED:",result,flush=True)

worker_results_df=pd.DataFrame(worker_results)
display(worker_results_df)

worker_results_df.to_csv(
    DOC_ROOT/"calibration_worker_returncodes.csv",
    index=False
)

bad=worker_results_df[
    worker_results_df["returncode"].fillna(-999)!=0
]

if len(bad):
    raise RuntimeError(
        "At least one theta trajectory failed. "
        "Do not select theta_DOC. Re-run to resume from checkpoints."
    )

## 6. Completude e seleção de \(\hat\theta_{DOC}\)

In [ ]:
expected_days=len(pd.date_range(CAL_START,CAL_END,freq="D"))
all_rows=[]
audit_rows=[]

for theta in THETA_GRID:
    outdir=CAL_ROOT/theta_label(theta)
    ck=outdir/"checkpoint.csv"
    st=outdir/"status.json"

    if not ck.exists() or not st.exists():
        raise RuntimeError(f"Missing outputs for theta={theta}")

    df=pd.read_csv(ck)
    status=json.loads(st.read_text(encoding="utf-8"))

    audit_rows.append({
        "theta":float(theta),
        "n_days":len(df),
        "expected_days":expected_days,
        "status":status.get("status"),
        "first_date":df["date"].iloc[0] if len(df) else None,
        "last_date":df["date"].iloc[-1] if len(df) else None,
    })

    if len(df)!=expected_days or status.get("status")!="completed":
        raise RuntimeError(
            f"Incomplete theta={theta}: "
            f"{len(df)}/{expected_days}, status={status.get('status')}"
        )

    all_rows.append(df)

calibration=pd.concat(all_rows,ignore_index=True)
audit=pd.DataFrame(audit_rows)
display(audit)

performance=(
    calibration.groupby("theta",as_index=False)
    .agg(
        mean_realized_cost=("total_realized_cost","mean"),
        median_realized_cost=("total_realized_cost","median"),
        cumulative_realized_cost=("total_realized_cost","sum"),
        mean_planned_cost=("planned_cost","mean"),
        total_load_shedding=("load_shedding","sum"),
        median_solve_seconds=("solve_seconds","median"),
        max_solve_seconds=("solve_seconds","max"),
    )
    .sort_values(["mean_realized_cost","theta"])
    .reset_index(drop=True)
)

performance["rank"]=np.arange(1,len(performance)+1)

min_cost=performance["mean_realized_cost"].min()
ties=performance[
    np.isclose(
        performance["mean_realized_cost"],
        min_cost,
        rtol=0,
        atol=1e-9
    )
].copy()

ties["abs_theta"]=ties["theta"].abs()
theta_doc=float(
    ties.sort_values(["abs_theta","theta"]).iloc[0]["theta"]
)

display(performance)

audit.to_csv(DOC_ROOT/"calibration_completeness_audit.csv",index=False)
calibration.to_csv(DOC_ROOT/"doc_calibration_day_level.csv",index=False)
performance.to_csv(DOC_ROOT/"doc_calibration_performance.csv",index=False)

print("SELECTED theta_DOC =",theta_doc)

## 7. Freeze do parâmetro DOC

In [ ]:
baseline_row=performance.loc[np.isclose(performance["theta"],0.0)].iloc[0]
doc_row=performance.loc[np.isclose(performance["theta"],theta_doc)].iloc[0]

theta_freeze={
    "theta_doc":theta_doc,
    "theta_baseline":0.0,
    "protocol_sha256":protocol_sha,
    "calibration_start":str(CAL_START.date()),
    "calibration_end":str(CAL_END.date()),
    "test_start":str(TEST_START.date()),
    "test_end":str(TEST_END.date()),
    "test_opened":False,
    "selection_metric":"mean_total_realized_cost",
    "selection_direction":"minimize",
    "tie_break":"smallest_abs_theta_then_theta",
    "theta_grid":[float(x) for x in THETA_GRID],
    "doc_mean_calibration_cost":float(doc_row["mean_realized_cost"]),
    "baseline_mean_calibration_cost":float(baseline_row["mean_realized_cost"]),
    "doc_minus_baseline_calibration":float(
        doc_row["mean_realized_cost"]-baseline_row["mean_realized_cost"]
    ),
}

freeze_file=DOC_ROOT/"theta_doc_freeze.json"
freeze_file.write_text(json.dumps(theta_freeze,indent=2),encoding="utf-8")

freeze_sha=hashlib.sha256(freeze_file.read_bytes()).hexdigest()
(DOC_ROOT/"theta_doc_freeze_sha256.txt").write_text(freeze_sha+"\n",encoding="utf-8")

print(json.dumps(theta_freeze,indent=2))
print("Freeze SHA256:",freeze_sha)

# STOP RULE

Ao terminar, envie:

```text
data/fame_energy_doc_v21/
```

Arquivos principais:

```text
doc_calibration_performance.csv
doc_calibration_day_level.csv
calibration_completeness_audit.csv
calibration_worker_returncodes.csv
theta_doc_freeze.json
theta_doc_freeze_sha256.txt
```

Se algum worker falhar, **não apague os checkpoints**. Execute novamente o notebook para retomar.

O Test permanece selado até o Notebook 06.